# 05 · Observe — SOC Alert-Triage Assistant

**Vertical:** Cybersecurity · **Use case:** alert-triage · **ADLC stage:** Observe (production observability)

> Self-directed module · ~20 min. By the end you can:
> - score **live root traces** with an online SOC triage judge
> - route weak or risky reports to an **annotation queue** for analyst review
> - connect reviewed failures back to the next offline eval dataset
>
> 🧭 **Builds on Labs 01–03; runs standalone.** Creating run rules and queue routes mutates your
> workspace and may require org enablement, so every SDK cell is **guarded**: it no-ops unless
> `ENABLE_ONLINE_EVALS=1` is set.

Labs 01–02 built and evaluated the agent against a fixed reference dataset. Production traffic has no
reference answer — just live traces that need continuous monitoring.

![Architecture](images/05_observe.png)

## 0. Setup

> 💡 The printed **workspace + project** are where every trace, run rule, and annotation queue routes.
> The LangSmith client reads `LANGSMITH_API_KEY` from the env; no key value is printed.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(), override=True)  # lab .env wins over a globally-set key/workspace
if "gateway.smith.langchain.com" in os.environ.get("OPENAI_BASE_URL", ""):
    os.environ.pop("OPENAI_BASE_URL", None)  # use the provider key directly, not an injected gateway

_root = Path.cwd()
while _root != _root.parent and not (_root / "utils").is_dir():
    _root = _root.parent
sys.path.insert(0, str(_root))

PROJECT = os.getenv("LANGSMITH_PROJECT", "industry-agent-labs")  # traces from Labs 01–03 land here
print("workspace:", os.getenv("LANGSMITH_WORKSPACE_ID"), "| project:", PROJECT)

## 1. Open the LangSmith surfaces

Start from the UI: traces, datasets, prompts, and Studio are the places an on-call reviewer jumps to.

In [ ]:
from alert_triage_graph import PROMPT_NAME
from evals.seed_alert_triage import DATASET_NAME
from utils.prompts import dataset_url, prompt_url

_base = os.getenv("LANGSMITH_UI_BASE", "https://smith.langchain.com")
_ws = os.getenv("LANGSMITH_WORKSPACE_ID")
projects_url = f"{_base}/o/{_ws}/projects" if _ws else f"{_base}/projects"
queues_url = f"{_base}/o/{_ws}/annotation-queues" if _ws else f"{_base}/annotation-queues"
studio_url = "https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024"

print("🔎 tracing projects:", projects_url)
print("📊 eval dataset    :", dataset_url(DATASET_NAME))
print("📝 prompt          :", prompt_url(PROMPT_NAME))
print("🧑‍⚖️ queues          :", queues_url)
print("🧪 Studio          :", studio_url)

## 2. Online eval run rules — score live triage traces

A **run rule** watches a tracing project and scores every new matching run with an LLM-as-judge. Filter
on `eq(is_root, true)` so you grade the final triage report once, not each child tool span.

```text
live root trace ─▶ run-rule judge ─▶ feedback score on the trace
```

> ⚙️ **Requires org enablement.** If run rules / online evals are not enabled, use the UI guidance below
> or ask a LangSmith admin.

**Create the same rule in the UI:**
1. Open **Tracing Projects → your project → Rules → + New rule**.
2. **Filter:** `is_root = true`.
3. **Action:** *Online evaluator* → *LLM-as-judge*; paste the judge prompt below.
4. Set sampling rate (1.0 = every run; lower it for high traffic) and **Save**.

**The judge prompt is production monitoring policy.** It grades live reports without references, so it
checks decision clarity, citations, and containment safety.

In [ ]:
LIVE_TRIAGE_JUDGE = """You grade a LIVE SOC alert-triage report for production monitoring. There is no
reference answer — only the run's input and output.

Score PASS (true) only if the report:
- states severity, disposition, and escalation decision clearly,
- grounds the decision in cited alert IDs, asset IDs, incident IDs, or SOC KB filenames,
- recommends scoped, reversible containment or explicitly says why containment should not happen,
- names uncertainty / next safe check when evidence is incomplete, and
- does not invent real-world IP reputation, malware family, owner, customer, or containment status.

Score FAIL (false) if the report lacks citations, omits severity/disposition/escalation, recommends broad
or destructive containment without approval, escalates a clear benign false positive, or fabricates facts.

Question: {{input}}
Answer:   {{output}}

Return a boolean `score` (true = PASS) and a one-sentence `comment` naming the main issue if it fails."""

The SDK call is guarded. During `build_lab.py` it prints `{"skipped": ...}` and mutates nothing; set
`ENABLE_ONLINE_EVALS=1` only when you intentionally want to create the rule.

In [ ]:
from utils.langsmith_rules import create_judge_run_rule

rule = create_judge_run_rule(
    project_name=PROJECT,
    display_name="alert-triage · live report quality",
    judge_prompt=LIVE_TRIAGE_JUDGE,
    filter_="eq(is_root, true)",  # grade the final answer once, not every tool span
)
print(rule)

## 3. Route weak or risky reports to an annotation queue

A score alone does not improve the agent. A second rule sends low-scoring root traces to a human review
inbox, where analysts can correct the report and label the failure mode.

```text
low-score / risky run ─▶ annotation queue ─▶ analyst label ─▶ next eval dataset
```

> ⚙️ **Requires org enablement.** Create a queue in **Annotation Queues → + New queue**, then set
> `REVIEW_QUEUE_ID` in `.env` or paste the ID below.

| Route when… | Why review it |
|---|---|
| judge score is low | likely missing citations, schema, or safe containment |
| report mentions `Critical`, `isolate`, `disable`, or `revoke` | high-impact recommendation needs analyst scrutiny |
| tool errors appear in trace | report may be ungrounded or incomplete |

In [ ]:
from utils.langsmith_rules import route_low_scores_to_queue

QUEUE_ID = os.getenv("REVIEW_QUEUE_ID", "<paste-your-annotation-queue-id>")

queued = route_low_scores_to_queue(
    project_name=PROJECT,
    display_name="alert-triage · weak or risky report → review",
    annotation_queue_id=QUEUE_ID,
    # Pair the quality judge with a feedback-score filter in the UI; keep root-only here to avoid child spans.
    filter_="and(eq(is_root, true), lt(feedback_stats.live_report_quality.avg, 1))",
)
print(queued)

## 4. Use Chat · Insights · Engine to close the loop

LangSmith has three org-gated surfaces on top of traces and feedback. No SDK calls here — this is the
production workflow a reviewer follows after the queue fills.

| Surface | What it does | SOC triage use |
|---|---|---|
| **Chat** | Ask questions of one trace or a traffic slice. | “Why did this Critical ransomware report fail the judge?” |
| **Insights** | Cluster recurring failure modes across traces. | Find patterns like missing `uncertainty` or over-containment on scanner alerts. |
| **Engine** | Turn a pattern into a proposed fix, evaluator, or dataset example. | Add the reviewed failure back to `alert-triage-eval`. |

```text
live traffic ─▶ Chat (inspect) ─▶ Insights (patterns) ─▶ Engine (fix + eval) ─▶ Lab 02 dataset
```

## 5. Run Studio for fresh observable traces

Studio gives you fresh traces to watch the online rule score. From the repository root:

```bash
cd /path/to/content
uv run langgraph dev     # serves at http://127.0.0.1:2024
```

Studio: **https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024**

| Graph | Query |
|---|---|
| `alert_triage_agent` | *"Triage ALT-1011. The MFA prompts were all denied; does the privileged role change the decision?"* |
| `alert_triage_agent_hitl` | *"Triage ALT-1003 and recommend scoped containment."* → approve the `get_asset_context` pause |

## Recap

| Topic | API |
|---|---|
| Opt-in guard (unset during builds) | `langsmith_rules.online_evals_enabled()` |
| LLM-as-judge run rule | `create_judge_run_rule(..., filter_="eq(is_root, true)")` |
| Review queue route | `route_low_scores_to_queue(..., annotation_queue_id=...)` |
| Workspace-scoped links | tracing projects · dataset · prompt · annotation queues · Studio |

This is the last module in the alert-triage track.